In [0]:
from pyspark.sql import functions as F
storage_account = "practicestg"
container = "data-lake"
base = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

df = spark.read.parquet(f"{base}/processed/")
df.printSchema()
display(df.limit(10))

root
 |-- Row_ID: string (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Ship_Date: string (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: string (nullable = true)



Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0,41.9136
2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0,219.582
3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0,6.8714
4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0,14.1694
7,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0,1.9656
8,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0,34.47


In [0]:
clean = (
    df
    .withColumn("Order_Date", F.to_date("Order_Date", "M/d/yyyy"))
    .withColumn("Ship_Date", F.to_date("Ship_Date", "M/d/yyyy"))
    .withColumn("Sales", F.col("Sales").cast("double"))
    .withColumn("Profit", F.col("Profit").cast("double"))
    .withColumn("Quantity", F.col("Quantity").cast("int"))
    .withColumn("Discount", F.col("Discount").cast("double"))
    .dropDuplicates(["Row_ID"])
    .withColumn("Order_Year", F.year("Order_Date"))
    .withColumn("Order_Month", F.month("Order_Date"))
    .withColumn("Ship_Days", F.datediff("Ship_Date", "Order_Date"))
) 

In [0]:
sales_by_category = (
    clean.groupBy("Category", "Sub-Category")
    .agg(F.round(F.sum("Sales"), 2).alias("Total_Sales"),
         F.round(F.sum("Profit"), 2).alias("Total_Profit"),
         F.sum("Quantity").alias("Total_Quantity"))
)

sales_by_region_year = (
    clean.groupBy("Region", "Order_Year")
    .agg(F.round(F.sum("Sales"), 2).alias("Total_Sales"),
         F.round(F.sum("Profit"), 2).alias("Total_Profit"))
)

In [0]:
sales_by_category.write.mode("overwrite").parquet(f"{base}/curated/sales_by_category")
sales_by_region_year.write.mode("overwrite").parquet(f"{base}/curated/sales_by_region_year")

In [0]:
display(dbutils.fs.ls(f"{base}/curated/"))

path,name,size,modificationTime
abfss://data-lake@practicestg.dfs.core.windows.net/curated/sales_by_category/,sales_by_category/,0,1793779270000
abfss://data-lake@practicestg.dfs.core.windows.net/curated/sales_by_region_year/,sales_by_region_year/,0,1793779277000


In [0]:
catalog = "dbx_deproject_01"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.superstore")

sales_by_category.write.mode("overwrite").saveAsTable(f"`{catalog}`.superstore.sales_by_category")
sales_by_region_year.write.mode("overwrite").saveAsTable(f"`{catalog}`.superstore.sales_by_region_year")

display(spark.sql(f"SELECT * FROM `{catalog}`.superstore.sales_by_category LIMIT 10"))

Category,Sub-Category,Total_Sales,Total_Profit,Total_Quantity
Office Supplies,Supplies,46673.54,-1189.1,647
Office Supplies,Fasteners,3024.28,949.52,914
Office Supplies,Binders,203412.73,30221.76,5974
Technology,Accessories,167380.32,41936.64,2976
Furniture,Chairs,328449.1,26590.17,2356
Technology,Phones,330007.05,44515.73,3289
Office Supplies,Paper,78479.21,34053.57,5178
Office Supplies,Appliances,107532.16,18138.01,1729
Office Supplies,Labels,12486.31,5546.25,1400
Furniture,Bookcases,114880.0,-3472.56,868
